# Progetto Smart Vehicular Systems - Multi-modal black box recorder
  - Gabriele Arcese
  - Diego Colì

## Setup

In [8]:
%pip install paho-mqtt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import carla
import os
import shutil
import time
import random
import socket
import threading
import queue
import numpy as np
import json
import paho.mqtt.client as mqtt
from dataclasses import dataclass, field, asdict
from typing import Optional
from collections import deque


try:
    import pygame
except ImportError:
    pygame = None
    
try:
    import cv2
    import numpy as np
except ImportError:
    cv2 = None
    np = None

## Connessione al client

In [50]:
client = carla.Client("localhost", 2000)
client.set_timeout(15.0)
world = client.get_world()
spectator = world.get_spectator()

print(f"Client version: {client.get_client_version()}")
print(f"Server version: {client.get_server_version()}")

Client version: 0.9.15
Server version: 0.9.15


## MQTT client

In [51]:
# queue thread-safe per accumulare messaggi ricevuti
mqtt_msgs_q = queue.Queue()
ring_buffer = deque(maxlen=600)  

def _on_connect(client, userdata, flags, rc):
    if rc == 0:
        client.connected_flag = True
        # sottoscrivi i topic desiderati
        client.subscribe([("recorder/control", 0), ("vehicle/+/status", 0)])
    else:
        client.connected_flag = False

def _on_message(client, userdata, msg):
    ts = time.time()
    payload = msg.payload
    # prova a decodificare json, fallback a stringa/bytes
    parsed = None
    try:
        parsed = json.loads(payload.decode("utf-8"))
    except Exception:
        try:
            parsed = payload.decode("utf-8")
        except Exception:
            parsed = payload  # raw bytes
    record = {"topic": msg.topic, "payload": parsed, "qos": msg.qos, "ts": ts}
    try:
        mqtt_msgs_q.put_nowait(record)
    except queue.Full:
        pass

def start_mqtt(broker="localhost", port=1883, keepalive=60):
    client = mqtt.Client()
    client.on_connect = _on_connect
    client.on_message = _on_message
    client.connected_flag = False
    client.connect(broker, port, keepalive=keepalive)
    client.loop_start()   # avvia loop in background thread
    return client

def stop_mqtt(client):
    if client is None: return
    try:
        client.loop_stop()
        client.disconnect()
    except Exception:
        pass

# Helper: svuota la queue e ritorna lista (da chiamare nel tick)
def drain_mqtt_queue():
    items = []
    while True:
        try:
            items.append(mqtt_msgs_q.get_nowait())
        except queue.Empty:
            break
    return items

## Dataclass EventRecord
Prodotto ad ogni tick della simulazione: è l'unità atomica del ring buffer.

| Campo | Sorgente CARLA | Tipo |
|---|---|---|
| `timestamp_sim` | `world.get_snapshot().timestamp.elapsed_seconds` | `float` |
| `timestamp_wall` | `time.time()` | `float` |
| `frame_id` | contatore incrementale | `int` |
| `camera_frames` | callback `sensor.camera.rgb` (una per camera) | `dict[str, np.ndarray]` |
| `location` | `vehicle.get_location()` | `carla.Location` |
| `velocity` | `vehicle.get_velocity()` | `carla.Vector3D` |
| `acceleration` | `vehicle.get_acceleration()` | `carla.Vector3D` |
| `control` | `vehicle.get_control()` | `carla.VehicleControl` |
| `warnings` | logica interna (LiDAR, collisione) | `list[str]` |
| `mqtt_messages` | thread MQTT | `list[str]` |
| `reasoning_text` | explanation agent | `str \| None` |
| `reasoning_ts` | wall-clock emissione agent | `float \| None` |

In [52]:
@dataclass
class VehicleState:
    """Stato cinematico e di controllo del veicolo in un istante."""
    x: float
    y: float
    z: float
    vx: float
    vy: float
    vz: float
    ax: float
    ay: float
    az: float
    throttle: float
    brake: float
    steer: float
    hand_brake: bool
    reverse: bool

    @property
    def speed_ms(self) -> float:
        """Velocità scalare in m/s."""
        return float(np.sqrt(self.vx**2 + self.vy**2 + self.vz**2))

    @property
    def speed_kmh(self) -> float:
        return self.speed_ms * 3.6

@dataclass
class EventRecord:
    """Unità atomica del ring buffer — un record per tick di simulazione."""
    # --- Temporali ---
    frame_id: int            # contatore tick, usato come chiave nel viewer
    timestamp_sim: float     # secondi simulazione (univoco per tick in sync mode)
    timestamp_wall: float    # wall-clock al momento della costruzione

    # --- Percezione ---
    vehicle_state: VehicleState

    # --- Camera (dict role → frame BGR; vuoto se nessun frame è ancora arrivato) ---
    # Esempio: {"front": np.ndarray, "rear": np.ndarray}
    camera_frames: dict = field(default_factory=dict, repr=False)

    # --- Logica e messaggistica ---
    warnings: list = field(default_factory=list)       # es. ["COLLISION_IMMINENT"]
    mqtt_messages: list = field(default_factory=list)  # payload raw ricevuti nel tick

    # --- Explanation agent ---
    reasoning_text: Optional[str] = None
    reasoning_ts: Optional[float] = None  # wall-clock di emissione dell'agente

    def to_dict(self) -> dict:
        """Serializza tutto tranne camera_frames (salvati separatamente come JPEG)."""
        d = asdict(self)
        d.pop("camera_frames")  # i frame vanno in frames/<frame_id:06d>_<role>.jpg
        return d

    def has_camera(self, role: str = None) -> bool:
        """True se almeno una camera (o la camera `role`) ha un frame nel tick."""
        if role:
            return role in self.camera_frames
        return len(self.camera_frames) > 0

    def camera_roles(self) -> list:
        return list(self.camera_frames.keys())

    def has_reasoning(self) -> bool:
        return self.reasoning_text is not None

Il **timestamp** di simulazione è la chiave di allineamento: tutto viene sincronizzato su di esso perché in modalità sincrona.

## VehicleState

In [53]:
# VehicleState partendo da un attore
def build_vehicle_state(vehicle) -> VehicleState:
    loc = vehicle.get_location()
    vel = vehicle.get_velocity()
    acc = vehicle.get_acceleration()
    ctrl = vehicle.get_control()
    return VehicleState(
    x=loc.x, y=loc.y, z=loc.z,
    vx=vel.x, vy=vel.y, vz=vel.z,
    ax=acc.x, ay=acc.y, az=acc.z,
    throttle=ctrl.throttle,
    brake=ctrl.brake,
    steer=ctrl.steer,
    hand_brake=ctrl.hand_brake,
    reverse=ctrl.reverse,
)

## Attivazione modalità sync
In modalità **sincrona**, il server avanza solo quando il client chiama `world.tick()`. Ciò è utile per esperimenti controllati e risultati riproducibili.


In [61]:
settings = world.get_settings()
settings.synchronous_mode = True
world.apply_settings(settings)
print("Modalità sincrona attivata")

Modalità sincrona attivata


## Spawn veicolo

In [55]:
def move_spectator_to(transform, spectator, distance=7.0, x=0, y=0, z=3.0, yaw=0, pitch=-15, roll=0):
    back_location = transform.location - transform.get_forward_vector() * distance
    
    back_location.x += x
    back_location.y += y
    back_location.z += z
    transform.rotation.yaw += yaw
    transform.rotation.pitch = pitch
    transform.rotation.roll = roll
    
    spectator_transform = carla.Transform(back_location, transform.rotation)
    
    spectator.set_transform(spectator_transform)

def spawn_vehicle(world, vehicle_index=20, spawn_index=10):
    blueprint_library = world.get_blueprint_library()
    vehicle_bp = blueprint_library.filter('vehicle.*')[vehicle_index]
    spawn_point = world.get_map().get_spawn_points()[spawn_index]
    vehicle = world.spawn_actor(vehicle_bp, spawn_point)
    return vehicle

def draw_on_screen(world, transform, content="O", color=carla.Color(0, 255, 0), life_time=20):
    world.debug.draw_string(transform.location, content, color=color, life_time=life_time)

def spawn_vehicle_ahead(world, ref_vehicle, distances=(20.0, 28.0, 36.0),
                        same_lane_first=True, retries=6):
    """Spawna un veicolo davanti al veicolo di riferimento."""
    ego_tf  = ref_vehicle.get_transform()
    ego_loc = ego_tf.location
    fwd     = ego_tf.get_forward_vector()
    bps     = world.get_blueprint_library().filter("vehicle.*")

    # Phase 1: offset diretto
    direct = sorted(set(float(d) for d in distances))
    direct += [d + 4.0 for d in direct]
    for d in direct:
        loc = ego_loc + fwd * d
        loc.z += 0.30
        actor = world.try_spawn_actor(random.choice(bps),
                                      carla.Transform(loc, ego_tf.rotation))
        if actor is not None:
            actor.set_autopilot(False)
            return actor

    # Phase 2: waypoint davanti
    road_map = world.get_map()
    ego_wp   = road_map.get_waypoint(ego_loc, project_to_road=True,
                                     lane_type=carla.LaneType.Driving)
    wp_candidates = []
    d_pool = sorted(set(float(d) for d in distances) |
                    set(float(d) + 8.0 for d in distances))
    for d in d_pool:
        try:
            cands = ego_wp.next(float(d))
        except RuntimeError:
            cands = []
        if same_lane_first:
            cands = sorted(cands, key=lambda w: (
                w.road_id != ego_wp.road_id,
                w.lane_id != ego_wp.lane_id,
                abs(w.lane_id - ego_wp.lane_id)))
        wp_candidates.extend(cands)

    for _ in range(max(1, int(retries))):
        for wp in wp_candidates:
            actor = world.try_spawn_actor(random.choice(bps), wp.transform)
            if actor is not None:
                actor.set_autopilot(False)
                return actor
        world.tick(); time.sleep(0.02)

    raise RuntimeError("Could not spawn or find a target vehicle ahead")

In [62]:
target = None
vehicle = spawn_vehicle(world)
vehicle.set_autopilot(False)  # Disattiva autopilot per controllo manuale

# Fai qualche tick per far stabilizzare il veicolo
for _ in range(10):
    world.tick()
    time.sleep(0.05)

# Ora spawna il target davanti
try:
    target = spawn_vehicle_ahead(world, vehicle, distances=(64.0, 20.0, 24.0), retries=15)
    print(f"Target spawned successfully")
except RuntimeError as e:
    target = None
    print(f"Target spawn failed: {e}")

# Se è spawnato, fermalo in posizione
if target is not None:
    target.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0, hand_brake=True))
    print("Target stopped with hand brake")

Flusher: saved 227 frames → recordings_20260512T133159
Flusher: saved 227 frames → recordings_20260512T133159
Flusher: saved 227 frames → recordings_20260512T133159
Flusher: saved 227 frames → recordings_20260512T133159
Flusher: saved 227 frames → recordings_20260512T133159
Flusher: saved 227 frames → recordings_20260512T133159
Flusher: saved 227 frames → recordings_20260512T133159
Flusher: saved 227 frames → recordings_20260512T133159
Flusher: saved 227 frames → recordings_20260512T133200
Flusher: saved 227 frames → recordings_20260512T133200
Target spawned successfully
Target stopped with hand brake


## Ciclo di simulazione

In [57]:
def tcp_check(host, port, timeout=2.0):
    s = socket.socket()
    s.settimeout(timeout)
    try:
        s.connect((host, port))
        s.close()
        return True, None
    except Exception as e:
        return False, str(e)

targets = [("localhost", 2001), ("test.mosquitto.org", 1883)]
for host, port in targets:
    ok, err = tcp_check(host, port)
    print(f"{host}:{port} -> {'OK' if ok else 'FAIL'}{'' if ok else ': '+err}")

# Try connecting with paho to localhost (shows MQTT error if refused)
client = mqtt.Client()
try:
    client.connect("localhost", 2001, keepalive=5)
    client.loop_start()
    time.sleep(0.3)
    client.loop_stop()
    print("paho-mqtt: connection to localhost:2001 succeeded")
except Exception as e:
    print("paho-mqtt: connection to localhost:2001 failed:", e)
    print("\nHints:")
    print("- If you want a local broker, start Mosquitto (outside notebook) or run Docker:")
    print("  docker run -d --name mosquitto -p 2001:2001 -p 9001:9001 eclipse-mosquitto")
    print("- Or switch to the public test broker: test.mosquitto.org:1883")

localhost:2001 -> OK
test.mosquitto.org:1883 -> FAIL: timed out


c:\anaconda3\envs\carla-env\lib\site-packages\ipykernel_launcher.py:17: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  app.launch_new_instance()


paho-mqtt: connection to localhost:2001 succeeded


## Collision trigger

In [ ]:

# --- Collision trigger + flusher setup ---
collision_event       = threading.Event()
collision_info        = {}
flusher_stop          = threading.Event()
stop_loop             = threading.Event()   # segnala al main loop di fermarsi
collision_frame_ready = threading.Event()   # segnala che il frame di collisione è nel buffer

def _on_collision(event):
    other   = getattr(event, "other_actor", None)
    impulse = getattr(event, "normal_impulse", None)
    collision_info.clear()
    collision_info.update({
        "other_actor": str(other.type_id) if other else "unknown",
        "impulse":     str(impulse),
        "ts":          time.time(),
    })
    collision_event.set()
    stop_loop.set()   # ferma il main loop

def flusher_thread(out_dir_base="recordings", retention_seconds=30):
    """Save only the last `retention_seconds` of frames (collision frame included)."""
    while not flusher_stop.is_set():
        if collision_event.wait(timeout=0.1):
            collision_event.clear()
            # Aspetta che il main loop abbia appendito il frame della collisione al buffer
            collision_frame_ready.wait(timeout=2.0)
            collision_frame_ready.clear()

            now = time.time()
            cutoff = now - float(retention_seconds)
            snapshot = [rec for rec in list(ring_buffer) if getattr(rec, 'timestamp_wall', now) >= cutoff]
            if not snapshot:
                snapshot = list(ring_buffer)[-1:]

            session_dir = f"{out_dir_base}_{time.strftime('%Y%m%dT%H%M%S')}"
            os.makedirs(session_dir, exist_ok=True)
            with open(os.path.join(session_dir, "events.jsonl"), "w", encoding="utf-8") as f:
                for rec in snapshot:
                    try:
                        json.dump(rec.to_dict(), f, default=str)
                    except Exception:
                        json.dump(asdict(rec) if hasattr(rec, '__dict__') else str(rec), f, default=str)
                    f.write("\n")

            metadata = {
                "trigger":        "collision",
                "collision_info": collision_info.copy(),
                "n_frames":       len(snapshot),
                "saved_at":       now,
                "retention_seconds": retention_seconds,
            }
            with open(os.path.join(session_dir, "metadata.json"), "w", encoding="utf-8") as f:
                json.dump(metadata, f, indent=2, default=str)

            # Cleanup old saved sessions older than retention_seconds
            try:
                parent_dir = os.getcwd()
                for name in os.listdir(parent_dir):
                    if not name.startswith(out_dir_base + "_"):
                        continue
                    dirpath = os.path.join(parent_dir, name)
                    if not os.path.isdir(dirpath):
                        continue
                    if now - os.path.getmtime(dirpath) > retention_seconds:
                        try:
                            shutil.rmtree(dirpath)
                        except Exception:
                            pass
            except Exception:
                pass

            print(f"Flusher: saved {len(snapshot)} frames → {session_dir}")


## Avvio sensore collisione e flusher

In [64]:
# --- Avvio sensore collisione e flusher ---
collision_sensor = None
try:
    col_bp = world.get_blueprint_library().find("sensor.other.collision")
    collision_sensor = world.spawn_actor(col_bp, carla.Transform(), attach_to=vehicle)
    collision_sensor.listen(_on_collision)
    flusher = threading.Thread(target=flusher_thread, args=("recordings",), daemon=True)
    flusher.start()
    print("Collision sensor e flusher avviati")
except Exception as e:
    print("Avvio collision sensor/flusher fallito:", e)

Collision sensor e flusher avviati


## Main simulation loop

In [ ]:

frame_id = 0
mqtt_client = None
try:
    stop_loop.clear()
    collision_frame_ready.clear()
    mqtt_client = start_mqtt(broker="localhost", port=2001)
    print("Simulazione avviata — ring buffer finestra 30 s")
    while True:
        world.tick()
        msgs = drain_mqtt_queue()
        # Applica controllo costante: accelerazione massima, senza frenata
        vehicle.apply_control(carla.VehicleControl(throttle=1.0, brake=0.0, steer=0.0))
        record = EventRecord(
            frame_id=frame_id,
            timestamp_sim=world.get_snapshot().timestamp.elapsed_seconds,
            timestamp_wall=time.time(),
            vehicle_state=build_vehicle_state(vehicle),
            camera_frames={},
            warnings=[],
            mqtt_messages=msgs,
        )
        ring_buffer.append(record)
        frame_id += 1

        # Se la collisione è avvenuta in questo tick, il frame è ora nel buffer
        if stop_loop.is_set():
            collision_frame_ready.set()  # sblocca il flusher: il frame di collisione è pronto
            print(f"Collisione rilevata al frame {frame_id} — stop simulazione, attendo flush…")
            time.sleep(1.5)   # lascia tempo al flusher_thread di completare il salvataggio
            break

        move_spectator_to(vehicle.get_transform(), spectator, distance=8.0, z=3.0, pitch=-20)
        time.sleep(0.05)
except KeyboardInterrupt:
    print("Interrotto manualmente")
finally:
    stop_mqtt(mqtt_client)


c:\anaconda3\envs\carla-env\lib\site-packages\ipykernel_launcher.py:32: DeprecationWarning: Callback API version 1 is deprecated, update to latest version


Flusher: saved 262 frames → recordings_20260512T133232
Flusher: saved 262 frames → recordings_20260512T133232
Flusher: saved 265 frames → recordings_20260512T133232
Flusher: saved 265 frames → recordings_20260512T133232
Flusher: saved 266 frames → recordings_20260512T133232
Flusher: saved 267 frames → recordings_20260512T133232
Flusher: saved 268 frames → recordings_20260512T133233
Flusher: saved 268 frames → recordings_20260512T133233
Flusher: saved 269 frames → recordings_20260512T133233
Flusher: saved 270 frames → recordings_20260512T133233
Flusher: saved 271 frames → recordings_20260512T133233
Flusher: saved 271 frames → recordings_20260512T133233
Flusher: saved 272 frames → recordings_20260512T133233
Flusher: saved 273 frames → recordings_20260512T133233
Flusher: saved 274 frames → recordings_20260512T133233
Flusher: saved 275 frames → recordings_20260512T133233
Flusher: saved 275 frames → recordings_20260512T133233
Flusher: saved 276 frames → recordings_20260512T133233
Flusher: s

In [66]:
settings.synchronous_mode = False
world.apply_settings(settings)
print("Modalità sincrona disattivata")

Modalità sincrona disattivata


Flusher: saved 600 frames → recordings_20260512T133322
Flusher: saved 600 frames → recordings_20260512T133322
Flusher: saved 600 frames → recordings_20260512T133322
Flusher: saved 600 frames → recordings_20260512T133322
Flusher: saved 600 frames → recordings_20260512T133322
Flusher: saved 600 frames → recordings_20260512T133322
Flusher: saved 600 frames → recordings_20260512T133322
Flusher: saved 600 frames → recordings_20260512T133322
Flusher: saved 600 frames → recordings_20260512T133322
Flusher: saved 600 frames → recordings_20260512T133322
Flusher: saved 600 frames → recordings_20260512T133323
Flusher: saved 600 frames → recordings_20260512T133323
Flusher: saved 600 frames → recordings_20260512T133323
Flusher: saved 600 frames → recordings_20260512T133323
Flusher: saved 600 frames → recordings_20260512T133323
Flusher: saved 600 frames → recordings_20260512T133323
Flusher: saved 600 frames → recordings_20260512T133323
Flusher: saved 600 frames → recordings_20260512T133323
Flusher: s

In [67]:
# --- Teardown: flusher + collision sensor + veicolo ---
try:
    flusher_stop.set()
except NameError:
    pass

if "collision_sensor" in dir() and collision_sensor is not None:
    try:
        collision_sensor.stop()
    except Exception:
        pass
    try:
        collision_sensor.destroy()
    except Exception:
        pass
    collision_sensor = None

# Distruggi il target se spawned
if target is not None:
    try:
        target.destroy()
    except Exception:
        pass
    target = None

vehicle.destroy()
print("Veicolo distrutto")
vehicle = None

Flusher: saved 600 frames → recordings_20260512T133516
Flusher: saved 600 frames → recordings_20260512T133516
Flusher: saved 600 frames → recordings_20260512T133517
Flusher: saved 600 frames → recordings_20260512T133517
Veicolo distrutto
